In [1]:
# [CELL 1] MISSION 1: VECTORIZED MEASUREMENT QUALITY ANALYSIS
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "CAJIDA"
STUDENT_ID = "TUPM-26-1243"
id_digits = "".join(d for d in STUDENT_ID if d.isdigit())

SEED_DIGIT = int(id_digits[-1])                      # 3
ID_SUM = sum(map(int, id_digits))                    # 18
RANDOM_SEED = int(id_digits) % (2**32)               # 261243

# Load SECOM dataset from parent data/ folder
data_dir = Path("../data")
output_dir = Path(".")
raw = np.genfromtxt(data_dir / "secom.data", dtype=float)

# Identify usable features (exclude all-missing and constant)
missing_mask = np.isnan(raw)
observed_counts = (~missing_mask).sum(axis=0)
all_missing = observed_counts == 0
obs_min = np.where(missing_mask, np.inf, raw).min(axis=0)
obs_max = np.where(missing_mask, -np.inf, raw).max(axis=0)
constant = (observed_counts > 0) & (obs_min == obs_max)
usable_idx = np.flatnonzero(~(all_missing | constant))

# Reproducibly select 5 + (SEED_DIGIT % 3) = 5 usable features
rng = np.random.default_rng(RANDOM_SEED)
n_select = 5 + (SEED_DIGIT % 3)
selected_idx = rng.choice(usable_idx, size=n_select, replace=False)
selected_names = [f"Feature_{i:03d}" for i in selected_idx]
sub_matrix = raw[:, selected_idx]

# Vectorized metrics
means = np.nanmean(sub_matrix, axis=0)
medians = np.nanmedian(sub_matrix, axis=0)
sds = np.nanstd(sub_matrix, axis=0, ddof=1)
missing_pct = 100.0 * np.isnan(sub_matrix).mean(axis=0)

# Robust standardization via Median Absolute Deviation (MAD)
abs_dev = np.abs(sub_matrix - medians)
mad = np.nanmedian(abs_dev, axis=0)
safe_mad = np.where(mad > 0, 1.4826 * mad, 1.0)
robust_z = np.where(mad > 0, (sub_matrix - medians) / safe_mad, 0.0)

flagged_mask = np.abs(robust_z) > 3.5
flagged_counts = np.nansum(flagged_mask, axis=0).astype(int)
flagged_pct = 100.0 * flagged_counts / sub_matrix.shape[0]

quality_df = pd.DataFrame({
    "Feature": selected_names,
    "Mean": np.round(means, 4),
    "Median": np.round(medians, 4),
    "Sample_SD": np.round(sds, 4),
    "Missing_Pct": np.round(missing_pct, 2),
    "Flagged_Count": flagged_counts,
    "Flagged_Pct": np.round(flagged_pct, 2)
}).sort_values(by="Flagged_Pct", ascending=False)

quality_df.to_csv(output_dir / "measurement_quality.csv", index=False)

print("Selected Features:", selected_names)
print("Highest Missing Percentage:", f"{quality_df['Missing_Pct'].max():.2f}%")
print("Feature with Highest Flagged Percentage:", quality_df.iloc[0]["Feature"])
print("Total Flagged Observations:", int(flagged_counts.sum()))
print("Vectorized Results Verified:", (output_dir / "measurement_quality.csv").is_file())

Selected Features: ['Feature_351', 'Feature_087', 'Feature_545', 'Feature_310', 'Feature_158']
Highest Missing Percentage: 91.19%
Feature with Highest Flagged Percentage: Feature_545
Total Flagged Observations: 998
Vectorized Results Verified: True


In [2]:
# [CELL 2] MISSION 2: GROUPED OUTCOME COMPARISON
labels = np.loadtxt(data_dir / "secom_labels.data", usecols=0, dtype=int)
feature_cols = [f"Feature_{i:03d}" for i in range(raw.shape[1])]
df_full = pd.DataFrame(raw, columns=feature_cols)
df_full.insert(0, "Example_ID", np.arange(len(df_full)))
df_full["Outcome"] = np.where(labels == -1, "Pass", "Fail")

# Full-dataset medians for imputation
usable_names = [f"Feature_{i:03d}" for i in usable_idx]
full_medians = df_full[usable_names].median()

# Reproducible balanced sample: min(20 + ID_SUM, Pass_count, Fail_count) = 38 per group
rng2 = np.random.default_rng(RANDOM_SEED)
pass_df = df_full[df_full["Outcome"] == "Pass"]
fail_df = df_full[df_full["Outcome"] == "Fail"]
n_bal = min(20 + ID_SUM, len(pass_df), len(fail_df))  # 38 per outcome

bal_pass = pass_df.iloc[rng2.choice(len(pass_df), size=n_bal, replace=False)]
bal_fail = fail_df.iloc[rng2.choice(len(fail_df), size=n_bal, replace=False)]
balanced_sample = pd.concat([bal_pass, bal_fail], ignore_index=True)

# Select 5 usable features and apply full-dataset median imputation
chosen_5 = list(rng2.choice(usable_names, size=5, replace=False))
for col in chosen_5:
    balanced_sample[col] = balanced_sample[col].fillna(full_medians[col])

balanced_sample[["Example_ID", "Outcome"] + chosen_5].to_csv(
    output_dir / "balanced_sample.csv", index=False
)

# Grouped statistics and Standardized Mean Difference (SMD)
comp_rows = []
for col in chosen_5:
    p_vals = balanced_sample.loc[balanced_sample["Outcome"] == "Pass", col]
    f_vals = balanced_sample.loc[balanced_sample["Outcome"] == "Fail", col]
    
    p_mean, f_mean = p_vals.mean(), f_vals.mean()
    p_sd, f_sd = p_vals.std(ddof=1), f_vals.std(ddof=1)
    
    pooled_sd = np.sqrt(((len(p_vals) - 1) * p_sd**2 + (len(f_vals) - 1) * f_sd**2) / (len(p_vals) + len(f_vals) - 2))
    abs_diff = abs(f_mean - p_mean)
    smd = abs_diff / pooled_sd if pooled_sd > 0 else 0.0
    
    comp_rows.append({
        "Feature": col,
        "Pass_Mean": round(p_mean, 4),
        "Fail_Mean": round(f_mean, 4),
        "Abs_Mean_Diff": round(abs_diff, 4),
        "Abs_SMD": round(smd, 4)
    })

comp_df = pd.DataFrame(comp_rows).sort_values(by="Abs_SMD", ascending=False)
comp_df.to_csv(output_dir / "feature_comparison.csv", index=False)

# Verify NumPy vs Pandas SD match on first chosen feature
np_sd_check = np.std(balanced_sample.loc[balanced_sample["Outcome"] == "Pass", chosen_5[0]].to_numpy(), ddof=1)
pd_sd_check = balanced_sample.loc[balanced_sample["Outcome"] == "Pass", chosen_5[0]].std(ddof=1)

print("Balanced Sample Size:", len(balanced_sample))
print("Selected Features:", chosen_5)
print("Highest-Ranked Feature:", comp_df.iloc[0]["Feature"])
print("Absolute Standardized Mean Difference:", f"{comp_df.iloc[0]['Abs_SMD']:.4f}")
print("NumPy-Pandas Statistics Match:", bool(np.isclose(np_sd_check, pd_sd_check)))

Balanced Sample Size: 76
Selected Features: [np.str_('Feature_118'), np.str_('Feature_389'), np.str_('Feature_574'), np.str_('Feature_473'), np.str_('Feature_050')]
Highest-Ranked Feature: Feature_389
Absolute Standardized Mean Difference: 0.2957
NumPy-Pandas Statistics Match: True


In [3]:
# [CELL 3] MISSION 3: STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS
valid_both = [
    col for col in usable_names
    if df_full.loc[df_full["Outcome"] == "Pass", col].notna().sum() >= 10
    and df_full.loc[df_full["Outcome"] == "Fail", col].notna().sum() >= 10
]

rng3 = np.random.default_rng(RANDOM_SEED)
robust_features = list(rng3.choice(valid_both, size=5, replace=False))
n_boot = 200 + 10 * SEED_DIGIT  # 230 repetitions

def run_bootstrap_analysis(seed_val):
    local_rng = np.random.default_rng(seed_val)
    results = []
    
    for col in robust_features:
        # Strategy 1: Available-case
        p_avail = df_full.loc[df_full["Outcome"] == "Pass", col].dropna().to_numpy()
        f_avail = df_full.loc[df_full["Outcome"] == "Fail", col].dropna().to_numpy()
        diff_avail = f_avail.mean() - p_avail.mean()
        
        boot_avail = [
            local_rng.choice(f_avail, size=len(f_avail), replace=True).mean() -
            local_rng.choice(p_avail, size=len(p_avail), replace=True).mean()
            for _ in range(n_boot)
        ]
        ci_avail_low, ci_avail_high = np.percentile(boot_avail, [2.5, 97.5])
        
        # Strategy 2: Full-dataset median imputation
        imputed_series = df_full[col].fillna(full_medians[col])
        p_imp = imputed_series[df_full["Outcome"] == "Pass"].to_numpy()
        f_imp = imputed_series[df_full["Outcome"] == "Fail"].to_numpy()
        diff_imp = f_imp.mean() - p_imp.mean()
        
        boot_imp = [
            local_rng.choice(f_imp, size=len(f_imp), replace=True).mean() -
            local_rng.choice(p_imp, size=len(p_imp), replace=True).mean()
            for _ in range(n_boot)
        ]
        ci_imp_low, ci_imp_high = np.percentile(boot_imp, [2.5, 97.5])
        
        results.append({
            "Feature": col,
            "Avail_Mean_Diff": round(diff_avail, 4),
            "Avail_Boot_Mean": round(float(np.mean(boot_avail)), 4),
            "Avail_CI_95": f"[{ci_avail_low:.4f}, {ci_avail_high:.4f}]",
            "Imp_Mean_Diff": round(diff_imp, 4),
            "Imp_Boot_Mean": round(float(np.mean(boot_imp)), 4),
            "Imp_CI_95": f"[{ci_imp_low:.4f}, {ci_imp_high:.4f}]",
            "Abs_Diff_Change": round(abs(diff_imp - diff_avail), 4),
            "Sign_Changed": bool(np.sign(diff_avail) != np.sign(diff_imp))
        })
    return pd.DataFrame(results)

res_run1 = run_bootstrap_analysis(RANDOM_SEED)
res_run2 = run_bootstrap_analysis(RANDOM_SEED)
reproducible = res_run1.equals(res_run2)

# Compare feature ranking by absolute mean difference
rank_avail = list(res_run1.reindex(res_run1["Avail_Mean_Diff"].abs().sort_values(ascending=False).index)["Feature"])
rank_imp = list(res_run1.reindex(res_run1["Imp_Mean_Diff"].abs().sort_values(ascending=False).index)["Feature"])
changed_ranks = sum(a != b for a, b in zip(rank_avail, rank_imp))

res_run1.to_csv(output_dir / "robustness_comparison.csv", index=False)

report_text = (
    "STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT\n"
    f"Student: {LAST_NAME} ({STUDENT_ID})\n"
    f"Selected Features: {', '.join(robust_features)}\n"
    f"Bootstrap Repetitions: {n_boot}\n"
    f"Largest Change in Mean Difference: {res_run1['Abs_Diff_Change'].max():.4f}\n"
    f"Features with Changed Difference Sign: {int(res_run1['Sign_Changed'].sum())}\n"
    f"Features with Changed Ranking: {changed_ranks}\n"
    f"Reproducibility Verified: {reproducible}\n"
)
(output_dir / "sensitivity_report.txt").write_text(report_text, encoding="utf-8")

print(report_text)

STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT
Student: CAJIDA (TUPM-26-1243)
Selected Features: Feature_352, Feature_086, Feature_545, Feature_311, Feature_159
Bootstrap Repetitions: 230
Largest Change in Mean Difference: 5.1183
Features with Changed Difference Sign: 0
Features with Changed Ranking: 0
Reproducibility Verified: True

